<a href="https://colab.research.google.com/github/Moussa018/Do-it-all-/blob/main/MultiHeadLatentAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import torch
import torch.nn as nn

class MultiHeadLatentAttention(nn.Module):
  def __init__(self , d_model = 128*128 , num_heads = 128 , q_latent_dimension = 12 , kv_latent_dimension = 4):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.q_latent_dimension = q_latent_dimension
    self.kv_latent_dimension = kv_latent_dimension
    head_dim = d_model // num_heads
    #Query projection
    self.Wq_d = nn.Linear(d_model , q_latent_dimension)
    #Precomputed matrix multiplications W_qU , W_kU for multiple heads
    self.W_qk = nn.Linear(q_latent_dimension , num_heads * kv_latent_dimension)
    #Key/Value latent projections
    self.Wkv_d = nn.Linear(d_model , kv_latent_dimension)
    self.Wv_u = nn.Linear(kv_latent_dimension , num_heads * head_dim)

    #Output projection
    self.Wo = nn.Linear(num_heads * head_dim , d_model)



In [ ]:
def forward(self , x):
    batch_size , seq_len , d_model = x.shape
    # First, the input x is projected into C_q and C_kv, which are shared by all attention heads.
    #First, you use self.W_qk to multiply C_q, then reshape the result into (batch_size, seq_len, num_heads, kv_latent_dim)
    C_q = self.Wq_d(x)
    #Then you multiply it with C_kv, after appropriate axis transpositions, to get the attention score.
    C_kv = self.Wkv_d(x)

    C_qW_qk = self.W_qk(C_q).view(batch_size , seq_len , self.num_heads , self.kv_latent_dimension)
    # Since C_qW_qk is a 4-dimensional tensor and C_kv is a 3-dimensional one, you add a dummy dimension to C_kv in place of the num_heads dimension.
    scores = torch.matmul(C_qW_qk.transpose(1,2) , C_kv.transpose(-2,-1)[:,None,...])/ math.sqrt(self.kv_latent_dim)
   # Attention computation
    att_weight = torch.softmax(scores , dim =-1)

    V = self.Wv_u(C_kv).view(batch_size, seq_len , self.num_heads,-1)
    # To get the attention output, you multiply the attention weight with V, which is computed from C_kv projected using self.Wv_u
    output = torch.matmul(att_weight , V.transpose(1,2).transpose(1,2),contiguous())
    #Finally, you concatenate the outputs of all heads and apply the output projection to get the final output.
    output = self.Wo(output.view(batch_size , seq_len , -1))
    return output


The original MLA paper suggests that it outperforms GQA in both model quality and inference speed. Since the matrices are smaller in this case, it’s also more memory efficient. However, you don’t need to train a model specifically for MLA. You can also convert a model trained with traditional multi-head attention to MLA by factoring the projection matrices after training.

